# Guide Ranker Training

Fine-tunes [Qwen/Qwen3.5-2B](https://huggingface.co/Qwen/Qwen3.5-2B) as a regression model
to predict `iterations_to_reach` from (goal, guide) s-expression pairs.

Lower predicted iterations = better guide.

Run with: `uv run jupyter notebook ranker.ipynb`

In [ ]:
from pathlib import Path
# import warnings

import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt

# causal-conv1d can't build with current CUDA version mismatch (system 13.1 vs torch 12.8)
# warnings.filterwarnings("ignore", message="The fast path is not available")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")
display(f"Using device: {device}")

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────
import math

MODEL_NAME = "Qwen/Qwen3.5-2B"
MAX_LENGTH = 256  # max tokens per (goal, guide) pair
SAMPLE_N = 10_000  # number of training+val examples to subsample
VAL_FRAC = 0.1  # fraction of data for validation
BASE_BATCH_SIZE = 16  # reference batch size for LR scaling
BASE_LR = 2e-5  # LR at reference batch size
BATCH_SIZE = 128
LR = BASE_LR * math.sqrt(BATCH_SIZE / BASE_BATCH_SIZE)  # sqrt scaling
EPOCHS = 3
SEED = 42
LOG_DIR = Path("runs/ranker-tb-logs")

display(f"LR: {LR:.2e} (sqrt-scaled from {BASE_LR:.0e} @ bs={BASE_BATCH_SIZE})")
torch.manual_seed(SEED)

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────
RUNS_DIR = Path("../runs")
candidates = sorted(RUNS_DIR.glob("run-*.csv"), key=lambda p: p.stat().st_mtime)
if not candidates:
    raise FileNotFoundError(f"No run-*.csv found in {RUNS_DIR.resolve()}")
csv_file = candidates[-1]
display(f"Using: {csv_file}")

df = pl.read_csv(csv_file)
display(f"Full dataset shape: {df.shape}")

# Keep only reached guides (we need a valid iterations_to_reach target)
df = df.filter(pl.col("reached")).drop_nulls(subset=["iterations_to_reach"])
display(f"Reached guides: {len(df)}")

# Subsample
if len(df) > SAMPLE_N:
    df = df.sample(SAMPLE_N, seed=SEED)
display(f"Sampled: {len(df)}")

# Train/val split
n_val = max(1, int(len(df) * VAL_FRAC))
df = df.sample(fraction=1.0, seed=SEED)  # shuffle
val_df = df.head(n_val)
train_df = df.tail(len(df) - n_val)
display(f"Train: {len(train_df)}, Val: {len(val_df)}")
display("Target stats (train):")
train_df["iterations_to_reach"].describe()

In [ ]:
# ── Tokenizer ──────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def format_input(goal: str, guide: str) -> str:
    """Format a (goal, guide) pair as a text prompt for the model."""
    return f"Goal: {goal}\nGuide: {guide}"

In [ ]:
# ── Dataset ─────────────────────────────────────────────────────────────
class GuideDataset(Dataset):
    def __init__(self, data: pl.DataFrame):
        self.goals = data["goal"].to_list()
        self.guides = data["guide"].to_list()
        self.targets = data["iterations_to_reach"].cast(pl.Float32).to_list()

    def __len__(self):
        return len(self.goals)

    def __getitem__(self, idx):
        text = format_input(self.goals[idx], self.guides[idx])
        enc = tokenizer(
            text,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=False,
            return_tensors="pt",
        )
        n_tokens = enc["input_ids"].size(1)
        assert n_tokens <= MAX_LENGTH, (
            f"Sample {idx} has {n_tokens} tokens, exceeds MAX_LENGTH={MAX_LENGTH}. "
            f"Text: {text[:100]}..."
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "target": torch.tensor(self.targets[idx], dtype=torch.float32),
        }


train_ds = GuideDataset(train_df)
val_ds = GuideDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# Sanity check
sample = train_ds[0]
display(f"Input shape: {sample['input_ids'].shape}")
display(f"Target: {sample['target'].item()}")
display(
    f"Decoded: {tokenizer.decode(sample['input_ids'], skip_special_tokens=True)[:200]}"
)

In [ ]:
# ── Model: Qwen backbone + regression head ─────────────────────────────
class GuideRanker(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.backbone = AutoModelForCausalLM.from_pretrained(
            model_name,
            trust_remote_code=True,
            dtype=torch.bfloat16,
        )
        hidden_size = self.backbone.config.hidden_size
        # Freeze most of the backbone, only train last 2 layers + head
        for param in self.backbone.parameters():
            param.requires_grad = False
        for layer in self.backbone.model.layers[-2:]:
            for param in layer.parameters():
                param.requires_grad = True

        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1),
        )
        # Head in float32 for stable regression
        self.head.float()

    @torch.compile()
    def forward(self, input_ids, attention_mask):
        attention_mask = attention_mask.to(torch.long)
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        # Use last hidden state at the last non-pad token
        hidden = outputs.hidden_states[-1]  # (B, seq_len, hidden)
        # Find index of last non-pad token per sequence
        seq_lengths = attention_mask.sum(dim=1) - 1  # (B,)
        batch_idx = torch.arange(hidden.size(0), device=hidden.device)
        pooled = hidden[batch_idx, seq_lengths]  # (B, hidden)
        return self.head(pooled.float()).squeeze(-1)  # (B,)


model = GuideRanker(MODEL_NAME).to(device)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
display(
    f"Trainable: {n_trainable:,} / {n_total:,} ({100 * n_trainable / n_total:.1f}%)"
)

In [ ]:
# ── Training ───────────────────────────────────────────────────────────
from tqdm.auto import tqdm

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
loss_fn = nn.MSELoss()

train_losses = []
val_losses = []

writer = SummaryWriter(log_dir=str(LOG_DIR))
global_step = 0


def evaluate():
    model.eval()
    total_loss = 0.0
    n = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            targets = batch["target"].to(device)
            preds = model(ids, mask)
            total_loss += loss_fn(preds, targets).item() * len(targets)
            n += len(targets)
    return total_loss / n


epoch_bar = tqdm(range(EPOCHS), desc="Training", unit="epoch")
for epoch in epoch_bar:
    model.train()
    epoch_loss = 0.0
    n_samples = 0

    step_bar = tqdm(
        train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}", leave=False, unit="batch"
    )
    for step, batch in enumerate(step_bar):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        targets = batch["target"].to(device)

        optimizer.zero_grad()
        preds = model(ids, mask)
        loss = loss_fn(preds, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()), 1.0
        )
        optimizer.step()

        step_loss = loss.item()
        epoch_loss += step_loss * len(targets)
        n_samples += len(targets)
        global_step += 1

        writer.add_scalar("train/step_loss", step_loss, global_step)
        step_bar.set_postfix(loss=f"{epoch_loss / n_samples:.4f}")

    train_loss = epoch_loss / n_samples
    val_loss = evaluate()
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step()

    writer.add_scalar("train/epoch_loss", train_loss, epoch + 1)
    writer.add_scalar("val/epoch_loss", val_loss, epoch + 1)
    writer.add_scalar("train/lr", scheduler.get_last_lr()[0], epoch + 1)

    epoch_bar.set_postfix(
        train=f"{train_loss:.4f}",
        val=f"{val_loss:.4f}",
        lr=f"{scheduler.get_last_lr()[0]:.1e}",
    )

writer.flush()
display(f"\nTensorBoard logs at: {LOG_DIR.resolve()}")
display(f"View with: tensorboard --logdir {LOG_DIR}")

In [ ]:
# ── Training curves ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, EPOCHS + 1), train_losses, "o-", label="train MSE")
ax.plot(range(1, EPOCHS + 1), val_losses, "o-", label="val MSE")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Training curves")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# ── Evaluation: predictions vs actual ──────────────────────────────────
import numpy as np
from scipy.stats import spearmanr

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch in val_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        preds = model(ids, mask).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(batch["target"].numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

mse = np.mean((all_preds - all_targets) ** 2)
mae = np.mean(np.abs(all_preds - all_targets))
rho, rho_p = spearmanr(all_preds, all_targets)

display(f"Val MSE:  {mse:.4f}")
display(f"Val MAE:  {mae:.4f}")
display(f"Val RMSE: {np.sqrt(mse):.4f}")
display(f"Spearman ρ: {rho:.4f} (p={rho_p:.2e})")

# Log final eval metrics to TensorBoard
writer.add_scalar("val/final_mse", mse, EPOCHS)
writer.add_scalar("val/final_mae", mae, EPOCHS)
writer.add_scalar("val/final_rmse", np.sqrt(mse), EPOCHS)
writer.add_scalar("val/final_spearman", rho, EPOCHS)
writer.close()

In [ ]:
# ── Boxplot: predicted vs actual iterations ────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
unique_targets = sorted(set(all_targets.astype(int)))  # pyright: ignore[reportAttributeAccessIssue]
grouped = [all_preds[all_targets.astype(int) == t] for t in unique_targets]  # pyright: ignore[reportAttributeAccessIssue]
ax.boxplot(grouped, tick_labels=unique_targets, showfliers=True)
ax.set_xlabel("Actual iterations_to_reach")
ax.set_ylabel("Predicted iterations_to_reach")
ax.set_title(f"Predicted by Actual (Spearman ρ={rho:.3f}, MAE={mae:.3f})")
fig.tight_layout()
plt.show()

In [ ]:
# ── Save model ─────────────────────────────────────────────────────────
SAVE_DIR = Path("../runs/ranker-checkpoint")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": {
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
            "val_mse": float(mse),
            "val_mae": float(mae),
            "val_spearman": float(rho),  # pyright: ignore[reportArgumentType]
            "sample_n": SAMPLE_N,
            "epochs": EPOCHS,
        },
    },
    SAVE_DIR / "ranker.pt",
)
display(f"Saved to {SAVE_DIR / 'ranker.pt'}")